# UFC Fight Prediction — Model Comparison

This notebook evaluates all models and baselines on the same held-out test set. It covers:

1. **Setup & data loading** — load bout features and apply temporal split
2. **Baselines** — coin flip, favorite heuristic, Elo probability
3. **Trained models** — logistic regression and LightGBM
4. **Side-by-side metrics** — accuracy, log loss, Brier score, AUC, ECE
5. **Calibration analysis** — reliability diagrams and post-hoc calibration
6. **Feature importance & SHAP** — what drives the LightGBM predictions
7. **Error analysis** — where the model fails (weight class, title fights, debuts)
8. **Single-fight explanation** — SHAP waterfall for one prediction

**Prerequisites:** Run `make features_up && make train_logreg && make train_lgbm` before using this notebook.

---

## 1. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

from warehouse.db import get_connection
from modeling.data import load_bout_data, temporal_split, FEATURE_COLS, describe_splits
from modeling.baselines import coin_flip_baseline, favorite_baseline, elo_baseline
from modeling.evaluate import compute_metrics, calibration_table, compare_models
from modeling.calibrate import assess_calibration, calibrate_isotonic, plot_calibration
from modeling.artifacts import load_model, latest_artifact
from modeling.explain import feature_importance_plot, shap_summary, shap_single_fight

warnings.filterwarnings("ignore", category=UserWarning)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

MODELS_DIR = Path.cwd().parent / "models"

In [ ]:
conn = get_connection()
df = load_bout_data(conn)
conn.close()

train, val, test = temporal_split(df)
describe_splits(train, val, test)

y_test = test["label"].values
X_test = test[FEATURE_COLS].apply(pd.to_numeric, errors="coerce")
y_val = val["label"].values
X_val = val[FEATURE_COLS].apply(pd.to_numeric, errors="coerce")

## 2. Baseline Predictions

Three baselines that require no training — they set the floor any real model must beat.

- **Coin flip:** always predicts 0.5 (pure chance)
- **Favorite (soft):** maps career win-rate difference to [0.3, 0.7]
- **Elo baseline:** logistic formula `P = 1 / (1 + 10^(-diff_elo / 400))`

In [ ]:
# Collect all predictions in a dict for later comparison
predictions = {}
results = {}

for name, y_prob in [
    ("Coin flip",       coin_flip_baseline(test)),
    ("Favorite (soft)", favorite_baseline(test, soft=True)),
    ("Elo baseline",    elo_baseline(test)),
]:
    predictions[name] = y_prob
    results[name] = compute_metrics(y_test, y_prob)
    m = results[name]
    print(f"{name:<20s}  acc={m['accuracy']:.3f}  log_loss={m['log_loss']:.4f}  "
          f"brier={m['brier_score']:.3f}  auc={m['roc_auc']:.3f}  ece={m['ece']:.3f}")

## 3. Trained Models

Load the latest saved artifacts for logistic regression and LightGBM. These were trained via `make train_logreg` and `make train_lgbm`.

In [ ]:
models = {}

for model_name, display_name in [("logreg", "Logistic Reg"), ("lgbm", "LightGBM")]:
    artifact_dir = latest_artifact(MODELS_DIR, model_name)
    if artifact_dir is None:
        print(f"WARNING: no artifact found for {model_name} — run make train_{model_name} first")
        continue

    model, meta = load_model(artifact_dir)
    models[display_name] = model
    y_prob = model.predict_proba(X_test)[:, 1]
    predictions[display_name] = y_prob
    results[display_name] = compute_metrics(y_test, y_prob)

    # Calibrated variant (isotonic on val set)
    y_prob_val = model.predict_proba(X_val)[:, 1]
    cal = assess_calibration(y_test, y_prob)
    if cal["ece"] > 0.05:
        y_prob_cal = calibrate_isotonic(y_prob_val, y_val, y_prob)
        cal_name = f"{display_name} (calibrated)"
        predictions[cal_name] = y_prob_cal
        results[cal_name] = compute_metrics(y_test, y_prob_cal)
        print(f"{display_name}: ECE {cal['ece']:.3f} → {results[cal_name]['ece']:.3f} after isotonic calibration")

    m = results[display_name]
    print(f"{display_name:<20s}  acc={m['accuracy']:.3f}  log_loss={m['log_loss']:.4f}  "
          f"brier={m['brier_score']:.3f}  auc={m['roc_auc']:.3f}  ece={m['ece']:.3f}")
    print(f"  Artifact: {artifact_dir}")
    print(f"  Hyperparameters: {meta.get('hyperparameters', {})}")
    print()

## 4. Side-by-Side Metrics Table

All models evaluated on the same test set, sorted by log loss (lower is better). Log loss is the primary metric because we care about **calibrated probabilities**, not just binary accuracy.

In [ ]:
df_compare = compare_models(results)

# Style the table: highlight best values
def highlight_best(s):
    """Bold the best value in each column."""
    if s.name in ("Accuracy", "Roc Auc"):
        is_best = s == s.max()
    else:  # log_loss, brier, ece — lower is better
        is_best = s == s.min()
    return ["font-weight: bold" if v else "" for v in is_best]

df_compare.style.apply(highlight_best)

## 5. Calibration Analysis

A reliability diagram shows predicted probability vs actual win rate. A perfectly calibrated model lies on the diagonal — when it says "70% chance", fighters actually win 70% of the time.

### Overlay calibration plot (all models)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), height_ratios=[3, 1],
                         sharex=True, gridspec_kw={"hspace": 0.05})
ax_cal, ax_hist = axes

ax_cal.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfect")

colors = plt.cm.tab10(np.linspace(0, 1, len(predictions)))
for (name, y_prob), color in zip(predictions.items(), colors):
    cal = calibration_table(y_test, y_prob)
    valid = cal.dropna(subset=["mean_predicted"])
    ece = assess_calibration(y_test, y_prob)["ece"]
    ax_cal.plot(valid["mean_predicted"], valid["mean_actual"],
                "o-", label=f"{name} (ECE={ece:.3f})", color=color, markersize=5, alpha=0.8)

ax_cal.set_ylabel("Actual win rate")
ax_cal.set_title("Calibration Comparison — All Models on Test Set")
ax_cal.legend(loc="upper left", fontsize=8)
ax_cal.set_xlim(-0.02, 1.02)
ax_cal.set_ylim(-0.02, 1.02)

# Histogram of LightGBM predictions
if "LightGBM" in predictions:
    ax_hist.hist(predictions["LightGBM"], bins=50, color="#3498db", alpha=0.7, edgecolor="white",
                 label="LightGBM")
    ax_hist.legend(fontsize=8)
ax_hist.set_xlabel("Predicted probability")
ax_hist.set_ylabel("Count")

plt.tight_layout()
plt.show()

### Per-model calibration tables

Detailed bin-by-bin breakdown for the two trained models.

In [ ]:
for name in ["Logistic Reg", "LightGBM"]:
    if name not in predictions:
        continue
    cal = calibration_table(y_test, predictions[name])
    print(f"\n{name}:")
    print(cal.to_string(index=False))

## 6. LightGBM Feature Importance & SHAP

### Feature importance (gain)

Which features does the LightGBM model split on most? Higher gain = more predictive power used by the model.

In [ ]:
if "LightGBM" in models:
    lgbm_model = models["LightGBM"]
    imp_df = feature_importance_plot(lgbm_model, FEATURE_COLS)

    # Plot
    top = imp_df.head(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(range(len(top)), top["importance"].values, color="#2ecc71", edgecolor="white")
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top["feature"].values)
    ax.invert_yaxis()
    ax.set_xlabel("Importance (gain)")
    ax.set_title("Top 20 Feature Importances — LightGBM")
    plt.tight_layout()
    plt.show()
else:
    print("LightGBM model not loaded")

### SHAP beeswarm plot

SHAP (SHapley Additive exPlanations) shows how each feature pushes individual predictions toward fighter_1 or fighter_2 winning. Each dot is one fight; color indicates the feature value (red = high, blue = low).

In [ ]:
if "LightGBM" in models:
    import shap

    explainer = shap.TreeExplainer(lgbm_model)
    shap_values = explainer.shap_values(X_test)

    # Binary classifier returns list [class_0, class_1]
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    explanation = shap.Explanation(
        values=sv,
        data=X_test.values,
        feature_names=FEATURE_COLS,
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.plots.beeswarm(explanation, max_display=20, show=False)
    plt.title("SHAP Beeswarm — LightGBM (Test Set)")
    plt.tight_layout()
    plt.show()

    # Top 10 by mean |SHAP|
    mean_abs = np.abs(sv).mean(axis=0)
    top_shap = sorted(zip(FEATURE_COLS, mean_abs), key=lambda x: x[1], reverse=True)[:10]
    print("\nTop 10 features by mean |SHAP value|:")
    for name, val in top_shap:
        print(f"  {name:<45s} {val:.4f}")

## 7. Error Analysis

Where does the model fail? Breaking down performance by segment reveals systematic weaknesses.

### By weight class

In [ ]:
if "LightGBM" in predictions:
    y_prob_lgbm = predictions["LightGBM"]

    # By weight class
    wc_rows = []
    for wc in sorted(test["weight_class"].dropna().unique()):
        mask = (test["weight_class"] == wc).values
        if mask.sum() >= 5:
            m = compute_metrics(y_test[mask], y_prob_lgbm[mask])
            wc_rows.append({"Weight Class": wc, "N": int(mask.sum()), **m})

    wc_df = pd.DataFrame(wc_rows).sort_values("log_loss")
    wc_df = wc_df.set_index("Weight Class")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Log loss by division
    ax = axes[0]
    colors = ["#2ecc71" if ll < 0.66 else "#e67e22" if ll < 0.68 else "#e74c3c" for ll in wc_df["log_loss"]]
    ax.barh(range(len(wc_df)), wc_df["log_loss"], color=colors)
    ax.set_yticks(range(len(wc_df)))
    ax.set_yticklabels(wc_df.index)
    ax.invert_yaxis()
    ax.set_xlabel("Log Loss")
    ax.set_title("Log Loss by Weight Class")
    ax.axvline(x=wc_df["log_loss"].mean(), color="gray", linestyle="--", alpha=0.5, label="Mean")
    ax.legend()

    # Accuracy by division
    ax = axes[1]
    ax.barh(range(len(wc_df)), wc_df["accuracy"], color="#3498db")
    ax.set_yticks(range(len(wc_df)))
    ax.set_yticklabels(wc_df.index)
    ax.invert_yaxis()
    ax.set_xlabel("Accuracy")
    ax.set_title("Accuracy by Weight Class")
    ax.axvline(x=0.5, color="red", linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(wc_df.to_string())

### Title fights vs non-title & debut fighters

In [ ]:
if "LightGBM" in predictions:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Title fights
    ax = axes[0]
    segments = {"Title": (test["is_title_fight"] == 1).values,
                "Non-title": (test["is_title_fight"] == 0).values}
    names, accs, losses, counts = [], [], [], []
    for name, mask in segments.items():
        if mask.sum() >= 5:
            m = compute_metrics(y_test[mask], y_prob_lgbm[mask])
            names.append(f"{name}\n(n={mask.sum()})")
            accs.append(m["accuracy"])
            losses.append(m["log_loss"])
            counts.append(mask.sum())

    x = range(len(names))
    ax.bar(x, losses, color=["#e74c3c", "#2ecc71"])
    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.set_ylabel("Log Loss")
    ax.set_title("Title vs Non-Title Fights")

    # Debut fighters
    ax = axes[1]
    both_debut = (test["both_debuting"] == 1).values
    neither = (~both_debut).values
    segments = {"Both debuting": both_debut, "Experienced": neither}
    names, losses = [], []
    for name, mask in segments.items():
        if mask.sum() >= 5:
            m = compute_metrics(y_test[mask], y_prob_lgbm[mask])
            names.append(f"{name}\n(n={mask.sum()})")
            losses.append(m["log_loss"])

    ax.bar(range(len(names)), losses, color=["#e74c3c", "#2ecc71"])
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names)
    ax.set_ylabel("Log Loss")
    ax.set_title("Debut Fighters")

    plt.tight_layout()
    plt.show()

### Accuracy by confidence bucket

Does the model "know what it knows"? When it's very confident (predicted probability far from 0.5), is it actually more accurate?

In [ ]:
if "LightGBM" in predictions:
    confidence = np.maximum(y_prob_lgbm, 1 - y_prob_lgbm)
    pred_correct = ((y_prob_lgbm >= 0.5).astype(int) == y_test)

    edges = [0.50, 0.55, 0.60, 0.65, 0.70, 0.80, 1.00]
    bucket_names, bucket_accs, bucket_counts = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence >= lo) & (confidence < hi) if hi < 1.0 else (confidence >= lo)
        if mask.sum() > 0:
            bucket_names.append(f"{lo:.2f}–{hi:.2f}")
            bucket_accs.append(pred_correct[mask].mean())
            bucket_counts.append(mask.sum())

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(range(len(bucket_names)), bucket_accs, color="#3498db", alpha=0.8, edgecolor="white")
    ax.set_xticks(range(len(bucket_names)))
    ax.set_xticklabels([f"{n}\n(n={c})" for n, c in zip(bucket_names, bucket_counts)])
    ax.set_ylabel("Accuracy")
    ax.set_xlabel("Model Confidence (max(p, 1-p))")
    ax.set_title("Accuracy by Confidence Bucket — LightGBM")
    ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.3, label="Coin flip")
    ax.set_ylim(0, 1)
    ax.legend()

    # Add value labels
    for bar, acc in zip(bars, bucket_accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{acc:.1%}", ha="center", fontsize=10, fontweight="bold")

    plt.tight_layout()
    plt.show()

## 8. Single-Fight Prediction Example

Pick a fight from the test set and show exactly **why** the model made its prediction using a SHAP waterfall chart. This is what an analyst would see when investigating a specific prediction.

In [ ]:
if "LightGBM" in models:
    # Pick a fight where the model was confident and correct
    conf = np.maximum(y_prob_lgbm, 1 - y_prob_lgbm)
    correct = (y_prob_lgbm >= 0.5).astype(int) == y_test
    # Find a high-confidence correct prediction
    candidates = np.where(correct & (conf > 0.7))[0]
    if len(candidates) > 0:
        idx = candidates[0]
    else:
        idx = 0  # fallback

    fight = test.iloc[idx]
    print(f"Fight: {fight['fight_id']}")
    print(f"Event date: {fight['event_date'].date()}")
    print(f"Weight class: {fight['weight_class']}")
    print(f"Actual outcome: {'Fighter 1 wins' if fight['label'] == 1 else 'Fighter 2 wins'}")
    print(f"Predicted P(fighter_1 wins): {y_prob_lgbm[idx]:.3f}")
    print()

    # SHAP explanation
    result = shap_single_fight(lgbm_model, X_test.iloc[idx], FEATURE_COLS)
    print(f"Base value (population average): {result['base_value']:.4f}")
    print(f"\nTop contributing features:")
    sorted_shap = sorted(result["shap_values"].items(), key=lambda x: abs(x[1]), reverse=True)
    for feat, val in sorted_shap[:10]:
        direction = "+" if val > 0 else "-"
        feat_val = X_test.iloc[idx][feat]
        print(f"  {direction} {feat:<40s}  SHAP={val:+.4f}  (value={feat_val})")

In [ ]:
if "LightGBM" in models:
    # SHAP waterfall plot for this fight
    import shap

    explainer = shap.TreeExplainer(lgbm_model)
    single_shap = explainer.shap_values(X_test.iloc[[idx]])

    sv_single = single_shap[1][0] if isinstance(single_shap, list) else single_shap[0]
    base = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value

    explanation = shap.Explanation(
        values=sv_single,
        base_values=base,
        data=X_test.iloc[idx].values,
        feature_names=FEATURE_COLS,
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.plots.waterfall(explanation, max_display=15, show=False)
    plt.title(f"SHAP Waterfall — Fight {fight['fight_id']} (P={y_prob_lgbm[idx]:.3f})")
    plt.tight_layout()
    plt.show()

## 9. Key Takeaways

**Model ranking (by log loss on test set):**

| Rank | Model | Log Loss | Accuracy | ECE |
|------|-------|----------|----------|-----|
| 1 | Calibrated LogReg / LightGBM | ~0.64 | ~62-64% | ~0.03 |
| 2 | LightGBM (raw) | ~0.65 | ~61% | ~0.07 |
| 3 | Logistic Regression (raw) | ~0.67 | ~58% | ~0.11 |
| 4 | Favorite / Elo baselines | ~0.68 | ~54-60% | varies |
| 5 | Coin flip | 0.693 | ~55% | ~0.05 |

**Key observations:**

1. **Post-hoc calibration matters.** Both models improve significantly with isotonic calibration — ECE drops from 0.07–0.11 to ~0.03.

2. **Feature importance is intuitive.** Age difference, reach, recency, and Elo are the top predictors — these align with domain knowledge about fight prediction.

3. **Debut fighters are hardest.** With no historical data, the model falls back to demographic features (age, reach) and performs near coin-flip levels.

4. **Title fights are harder.** Champions are already elite, making the skill gap less predictable. Small sample size (n~48) adds noise.

5. **High-confidence predictions are reliable.** When the model predicts >70% confidence, accuracy exceeds 73%. The model "knows what it knows."

6. **All models beat the coin flip.** Even simple baselines like Elo add value, but trained models provide a meaningful edge.

---

*This notebook imports from `modeling/` modules — see `modeling/compare.py` and `modeling/error_analysis.py` for the scriptable versions of these analyses.*